<a href="https://colab.research.google.com/github/Goldeno10/flexisaf_Internship_GenAI_DS_Intermediate/blob/main/task_4/computer_vision_project/Image_Captioning_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pickle
import torch
from torch import nn
import numpy as np
from tqdm import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/adityajn105/flickr8k")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")
# path = kagglehub.dataset_download("adityajn105/flickr30k")

print("Path to dataset files:", path)

In [ ]:
import torchvision.transforms as transforms


# Define the transform pipeline
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],    # Standard ImageNet mean
        std=[0.229, 0.224, 0.225]      # Standard ImageNet std
    )
])


In [ ]:
import torch
from collections import Counter


class Vocabulary:
    """
    handles word-to-index mapping and should be saved as
    a .pkl file so you can swap it when changing datasets.
    """
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in sentence.lower().split():
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = text.lower().split()
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text]


In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class ImageCaptionDataset(Dataset):
    """Dataset with flexible loading for Flickr8k/30k/MS COCO."""
    def __init__(self, root_dir, captions_df, transform=None, vocab=None):
        self.root_dir = root_dir
        self.df = captions_df
        self.transform = transform
        self.vocab = vocab

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        icaption = self.df["caption"][index]
        img_id = self.df["image"][index]
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        # Numericalize caption and add start/end tokens
        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption += [self.vocab.stoi["<EOS>"]]

        return img, torch.tensor(numericalized_caption)



class MyCollate:
    """Pads batches dynamically."""
    def __init__(self, pad_idx): self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = torch.stack([item[0] for item in batch])
        targets = pad_sequence([item[1] for item in batch], batch_first=True, padding_value=self.pad_idx)
        return imgs, targets


In [ ]:
class EncoderCNN(nn.Module):
    """
    CNN Encoder Model

    We use a pre-trained ResNet50. We freeze the weights and
    replace the final FC layer with a linear layer that matches our hidden size.
    """
    def __init__(self, embed_size, train_CNN=False):
        super(EncoderCNN, self).__init__()
        self.train_CNN = train_CNN
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        features = self.resnet(images)
        return self.dropout(self.relu(features))


In [ ]:
class DecoderRNN(nn.Module):
    """This takes the image features and generates word sequences."""
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        # features shape: [batch, embed_size]
        # captions shape: [seq_len, batch]
        embeddings = self.dropout(self.embed(captions))

        # Add a time dimension to features to concatenate with caption embeddings
        # result: [seq_len + 1, batch, embed_size]
        embeddings = torch.cat((features.unsqueeze(0), embeddings), dim=0)

        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs


In [ ]:
class CNNtoRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(CNNtoRNN, self).__init__()
        self.encoderCNN = EncoderCNN(embed_size)
        self.decoderRNN = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoderCNN(images)
        outputs = self.decoderRNN(features, captions)
        return outputs


In [ ]:
def train():
    # Hyperparameters
    embed_size = 256
    hidden_size = 256
    num_layers = 1
    learning_rate = 3e-4
    num_epochs = 20

    # Initialize model, loss, optimizer
    model = CNNtoRNN(embed_size, hidden_size, vocab_size, num_layers).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=dataset.vocab.stoi["<PAD>"])
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        for idx, (imgs, captions) in enumerate(dataloader):
            imgs, captions = imgs.to(device), captions.to(device)

            outputs = model(imgs, captions[:-1]) # Don't pass <EOS> to decoder

            # Reshape for CrossEntropy: (seq_len * batch, vocab_size)
            loss = criterion(outputs.reshape(-1, outputs.shape[2]), captions.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
